# Retail E-commerce Sales Analytics Pipeline
## Phase 4: Gold Layer — Conformed Fact Table + Analytics Tables

This notebook builds:
- `fact_orders` — the conformed fact table, resolving **surrogate keys**
  from the SCD Type 2 dimensions using a **point-in-time join** (the order
  date must fall between the dimension row's `effective_start_date` and
  `effective_end_date`)
- 4 Gold analytics tables: `gold_daily_sales`, `gold_category_sales`,
  `gold_segment_sales`, `gold_region_sales`

In [0]:
CATALOG = "retail_demo"
RAW_SCHEMA = "raw"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Ready.")

Ready.


## 1. Load Silver Sources

In [0]:
silver1_orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean")
dim_customer_scd2 = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2")
dim_product_scd2 = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2")
dim_store = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_store")

# dim_store already has a permanent store_sk column (assigned in Silver Stage 1),
# so we just use it directly — no need to regenerate it here.
dim_store_sk = dim_store

print("silver1_orders:", silver1_orders.count())
print("dim_customer_scd2:", dim_customer_scd2.count())
print("dim_product_scd2:", dim_product_scd2.count())
print("dim_store_sk:", dim_store_sk.count())

silver1_orders: 17344
dim_customer_scd2: 3093
dim_product_scd2: 980
dim_store_sk: 75


## 2. Prepare Orders — Derive `order_date` for Point-in-Time Matching

In [0]:
orders_with_date = (
    silver1_orders
    .withColumn("order_date", F.to_date("order_ts_final"))
)

display(orders_with_date.select("order_id", "order_ts_final", "order_date", "customer_id", "product_id", "store_id").limit(5))

order_id,order_ts_final,order_date,customer_id,product_id,store_id
O0000001,2025-12-15T04:49:00.000Z,2025-12-15,C01671,P00638,S003
O0000002,2026-03-13T03:29:00.000Z,2026-03-13,C00713,P00768,S028
O0000003,2026-03-05T07:20:00.000Z,2026-03-05,C02131,P00142,S072
O0000004,2025-12-28T13:56:00.000Z,2025-12-28,C01399,P00031,S071
O0000005,2025-12-31T17:43:00.000Z,2025-12-31,C00383,P00564,S049


## 3. Point-in-Time Join — Resolve `customer_sk`
**This is the crucial join logic** the guide calls out: we don't just match
on `customer_id` — we also require the order's date to fall inside that
dimension row's active date range. This correctly picks up the version of
the customer that was true *at the time the order was placed*, even for
late-arriving orders.

In [0]:
orders_with_customer_sk = (
    orders_with_date.alias("o")
    .join(
        dim_customer_scd2.alias("dc"),
        (F.col("o.customer_id") == F.col("dc.customer_id")) &
        (F.col("o.order_date") >= F.col("dc.effective_start_date")) &
        (F.col("o.order_date") <= F.col("dc.effective_end_date")),
        how="left"
    )
    .select(
        "o.*",
        F.col("dc.surrogate_key").alias("customer_sk"),
        F.col("dc.segment").alias("customer_segment_at_sale"),
    )
)

unresolved_customers = orders_with_customer_sk.filter(F.col("customer_sk").isNull()).count()
print("Orders where customer_sk could NOT be resolved:", unresolved_customers)
print("Total orders:", orders_with_customer_sk.count())

Orders where customer_sk could NOT be resolved: 0
Total orders: 17344


In [0]:
display(
    orders_with_product_sk
    .filter(F.col("product_sk").isNull())
    .select("order_id", "product_id", "order_date")
    .limit(20)
)

order_id,product_id,order_date


## 4. Point-in-Time Join — Resolve `product_sk`

In [0]:
orders_with_product_sk = (
    orders_with_customer_sk.alias("o")
    .join(
        dim_product_scd2.alias("dp"),
        (F.col("o.product_id") == F.col("dp.product_id")) &
        (F.col("o.order_date") >= F.col("dp.effective_start_date")) &
        (F.col("o.order_date") <= F.col("dp.effective_end_date")),
        how="left"
    )
    .select(
        "o.*",
        F.col("dp.surrogate_key").alias("product_sk"),
        F.col("dp.category").alias("product_category_at_sale"),
    )
)

unresolved_products = orders_with_product_sk.filter(F.col("product_sk").isNull()).count()
print("Orders where product_sk could NOT be resolved:", unresolved_products)

Orders where product_sk could NOT be resolved: 0


## 5. Join — Resolve `store_sk`
Stores don't change over time in this dataset, so this is a simple join
(no date range needed) — but we still use the surrogate key, not the
natural `store_id`, as the foreign key in the fact table.

In [0]:
orders_with_store_sk = (
    orders_with_product_sk.alias("o")
    .join(
        dim_store_sk.select("store_id", "store_sk", "region").alias("ds"),
        F.col("o.store_id") == F.col("ds.store_id"),
        how="left"
    )
    .select(
        "o.*",
        F.col("ds.store_sk"),
        F.col("ds.region").alias("store_region_at_sale"),
    )
)

unresolved_stores = orders_with_store_sk.filter(F.col("store_sk").isNull()).count()
print("Orders where store_sk could NOT be resolved:", unresolved_stores)

Orders where store_sk could NOT be resolved: 0


## 6. Build & Save `fact_orders`

In [0]:
fact_orders = (
    orders_with_store_sk
    .select(
        "order_id", "order_date", "order_ts_final",
        "customer_sk", "product_sk", "store_sk",
        "customer_id", "product_id", "store_id",   # natural keys kept for debugging/traceability
        F.col("quantity_final").alias("quantity"),
        F.col("unit_price_final").alias("unit_price"),
        F.col("discount_pct_final").alias("discount_pct"),
        F.col("gross_amount_final").alias("gross_amount"),
        "payment_method", "order_status", "coupon_code",
        "customer_segment_at_sale", "product_category_at_sale", "store_region_at_sale",
    )
)

(fact_orders.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders"))

print("fact_orders row count:", spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders").count())
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders").limit(5))

fact_orders row count: 17344


order_id,order_date,order_ts_final,customer_sk,product_sk,store_sk,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,coupon_code,customer_segment_at_sale,product_category_at_sale,store_region_at_sale
O0000001,2025-12-15,2025-12-15T04:49:00.000Z,1671,638,3,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled,null,Gold,Home,South
O0000004,2025-12-28,2025-12-28T13:56:00.000Z,1399,31,71,C01399,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned,null,Regular,Beauty,Online
O0000007,2025-12-26,2025-12-26T03:16:00.000Z,1633,555,20,C01633,P00555,S020,2,54612.57,0.05,98302.63,COD,returned,null,Gold,Electronics,Online
O0000008,2025-12-09,2025-12-09T20:57:00.000Z,1960,771,29,C01960,P00771,S029,1,29725.2,0.0,28238.94,UPI,returned,null,Regular,Home,West
O0000009,2026-01-20,2026-01-20T19:25:00.000Z,479,782,50,C00479,P00782,S050,4,84092.97,0.05,336371.88,UPI,delivered,null,Silver,Beauty,East


## 7. Gold Analytics Tables

### 7a. Daily Sales

In [0]:
gold_daily_sales = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
        F.round(F.sum("gross_amount") / F.countDistinct("order_id"), 2).alias("avg_order_value"),
    )
    .orderBy("order_date")
)

(gold_daily_sales.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales"))

display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales").limit(10))

order_date,total_orders,total_revenue,total_units,avg_order_value
2025-10-01,60,7931198.779999998,193,132186.65
2025-10-02,49,5833780.930000002,135,119056.75
2025-10-03,57,6742787.45,160,118294.52
2025-10-04,64,8789390.049999999,201,137334.22
2025-10-05,53,6658672.450000001,159,125635.33
2025-10-06,61,8034282.170000002,188,131709.54
2025-10-07,63,8086661.389999999,174,128359.7
2025-10-08,63,6900786.519999997,185,109536.29
2025-10-09,49,6025594.720000001,136,122971.32
2025-10-10,65,7819608.320000002,208,120301.67


### 7b. Category Sales

In [0]:
gold_category_sales = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
    .groupBy(F.col("product_category_at_sale").alias("category"))
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
    )
    .orderBy(F.col("total_revenue").desc())
)

(gold_category_sales.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_category_sales"))

display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_category_sales"))

category,total_orders,total_revenue,total_units
Home,3987,5.0210854466000056E8,12595
Fashion,3773,4.821459476999986E8,11830
Electronics,3214,4.0610935583999974E8,10068
Grocery,3085,4.0408677799999934E8,9580
Beauty,2984,3.7448941568000007E8,9333
Unknown,289,3.780453427999997E7,884
null,12,2167305.27,43


### 7c. Segment Sales

In [0]:
gold_segment_sales = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
    .groupBy(F.col("customer_segment_at_sale").alias("segment"))
    .agg(
        F.countDistinct("customer_id").alias("unique_customers"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
    )
    .orderBy(F.col("total_revenue").desc())
)

(gold_segment_sales.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_segment_sales"))

display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_segment_sales"))

segment,unique_customers,total_orders,total_revenue
Platinum,696,4501,5.699300522600011E8
Regular,671,4416,5.63404869709999E8
Silver,657,4336,5.53466402620001E8
Gold,661,4091,5.221105568400004E8


### 7d. Region Sales

In [0]:
gold_region_sales = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
    .groupBy(F.col("store_region_at_sale").alias("region"))
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
    )
    .orderBy(F.col("total_revenue").desc())
)

(gold_region_sales.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_region_sales"))

display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_region_sales"))

region,total_orders,total_revenue,total_units
Online,4425,5.609071000899985E8,13726
South,3963,5.070504159599997E8,12406
North,3310,4.289448267200011E8,10494
West,3194,4.078078436800005E8,10135
East,2452,3.0420169498000056E8,7572


## 8. Delta Lake Feature Demonstration — History & Time Travel

In [0]:
%sql
SELECT * FROM retail_demo.gold.fact_orders VERSION AS OF 0 LIMIT 10;

order_id,order_date,order_ts_final,customer_sk,product_sk,store_sk,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,coupon_code,customer_segment_at_sale,product_category_at_sale,store_region_at_sale
O0000001,2025-12-15,2025-12-15T04:49:00.000Z,1671,638,3,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled,null,Gold,Home,South
O0000002,2026-03-13,2026-03-13T03:29:00.000Z,713,768,28,C00713,P00768,S028,4,30049.26,0.1,102167.48,CARD,returned,null,Silver,Grocery,East
O0000003,2026-03-05,2026-03-05T07:20:00.000Z,2131,142,72,C02131,P00142,S072,5,59685.88,0.0,283507.93,COD,delivered,null,Platinum,Electronics,North
O0000004,2025-12-28,2025-12-28T13:56:00.000Z,1399,31,71,C01399,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned,null,Regular,Beauty,Online
O0000005,2025-12-31,2025-12-31T17:43:00.000Z,383,564,49,C00383,P00564,S049,2,46727.14,0.15,84108.85,COD,cancelled,null,Gold,Grocery,West
O0000006,2026-03-11,2026-03-11T02:59:00.000Z,1745,101,19,C01745,P00101,S019,2,12387.77,0.05,21059.21,COD,returned,null,Gold,Fashion,South
O0000007,2025-12-26,2025-12-26T03:16:00.000Z,1633,555,20,C01633,P00555,S020,2,54612.57,0.05,98302.63,COD,returned,null,Gold,Electronics,Online
O0000008,2025-12-09,2025-12-09T20:57:00.000Z,1960,771,29,C01960,P00771,S029,1,29725.2,0.0,28238.94,UPI,returned,null,Regular,Home,West
O0000009,2026-01-20,2026-01-20T19:25:00.000Z,479,782,50,C00479,P00782,S050,4,84092.97,0.05,336371.88,UPI,delivered,null,Silver,Beauty,East
O0000010,2026-03-21,2026-03-21T06:52:00.000Z,197,68,49,C00197,P00068,S049,5,31466.94,0.0,157334.7,COD,delivered,null,Platinum,Fashion,West


In [0]:
%sql
DESCRIBE HISTORY retail_demo.gold.fact_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-11T15:35:58.000Z,71577276216912,jasmeetk7082@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1100778341048516),b7efec31-7c4a-4d1e-86df-0a9003855859,0711-135504-b7780eky-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 429936, numDeletionVectorsRemoved -> 0, numOutputRows -> 17344, numOutputBytes -> 431616)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-11T15:20:43.000Z,71577276216912,jasmeetk7082@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1100778341048516),f8d0eed8-96a2-4aa5-8840-fd28c54fc56f,0711-135504-b7780eky-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 284874, numDeletionVectorsRemoved -> 0, numOutputRows -> 17344, numOutputBytes -> 429936)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-11T14:51:15.000Z,71577276216912,jasmeetk7082@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1100778341048516),890f4ad4-1174-4bff-b263-d842c8d9b24d,0711-135504-b7780eky-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 11557, numOutputBytes -> 284874)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


## Summary — Phase 4 (Gold Layer)
- Built `fact_orders` by resolving **surrogate keys** (`customer_sk`,
  `product_sk`, `store_sk`) from the SCD Type 2 dimensions using a
  **point-in-time join**: each order's date must fall between the
  dimension row's `effective_start_date` and `effective_end_date`. This
  guarantees the fact table reflects the customer/product state *as it
  was at the time of the sale*, correctly handling late-arriving orders.
- Confirmed (via the unresolved-key counts) that all orders successfully
  resolved to a valid dimension version.
- Built 4 Gold analytics tables — `gold_daily_sales`, `gold_category_sales`,
  `gold_segment_sales`, `gold_region_sales` — all aggregated directly from
  `fact_orders`, ready for dashboards/reporting.
- All 18 tables from the project's "Expected Final Deliverables" list are
  now populated.